In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path
import logging
import pandas as pd
from datasets import Dataset
from flwr_datasets.partitioner import IidPartitioner
from datasets import Dataset, Sequence, Value, Features


**Nota:** no muevas el script de sitio o se rompe

# Preprocesado de datos

Este script nos va a permitir preparar los datasets para el entrenamiento del modelo, esto lo vamos a realizar de dos maneras diferentes: normalización y expresión en forma de línea temporal a través de secuencias.

La primero vamos a realizar una normalización de los datos y los colocaremos de tal manera que cada uno de los que los datos de entrada sean las lecturas de los sensores en las últimas 10 horas y la salida será la temperatura que va ha hacer en la siguiente hora. Esto require que el dataset sea de dos dimensiones, así que se guardará como JSON.

NOTA: aquí vamos a desglosar y adaptar la función para pleidata de la clase `DataLoader` original. Cambiando también la configuración por constantes.

In [8]:
def create_sequences(features, target, window_size):
    """Función base unificada para crear secuencias."""
    X_list, Y_list = [], []
    for i in range(len(features) - window_size):
        seq_x = features[i: i + window_size]
        seq_y = target[i + window_size]
        X_list.append(seq_x)
        Y_list.append(seq_y)
    return np.array(X_list), np.array(Y_list)

In [9]:
# 1. Definir Configuración
feature_cols = ['dif_cons_real', 
        'dif_cons_smooth', 
        'V2', 
        'V4', 
        'V12', 
        'V26', 
        'Hour_1', 
        'Hour_2', 
        'Hour_3', 
        'Season_1', 
        'Season_2', 
        'Season_3', 
        'Season_4', 
        'tmed', 
        'hrmed', 
        'radmed', 
        'vvmed', 
        'dvmed', 
        'prec', 
        'dewpt', 
        'dpv']
target_col: str = 'dif_cons_real'
window_size:int = 12
# El siguiente solo es necesario en caso de convlstm que necesita una entrada específica
model_type: str = "lstm" #"transformer" "lstm" "convlstm" "gru" "gruSimple"
dataset:str = 'data-model-consumoC-60T.csv'
output_file_name: str = 'buildingC-data.parquet'

In [10]:
# 2. Cargar Datos
base_dir = Path('..')
file_path =  base_dir / 'og' / dataset # NOTE: Si has movido el script arregla aquí
if not file_path.exists():
    raise FileNotFoundError(f"Fichero de datos no encontrado en {file_path}")

df = pd.read_csv(file_path, sep=';', index_col=0)

In [11]:
# 3. Limpieza (sin escalar aquí)
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.dropna(subset=feature_cols, inplace=True)

if len(df) <= window_size:
    raise ValueError(f"Datos insuficientes tras limpieza ({len(df)} filas) para window_size ({window_size})")

In [12]:
# 4. Escalado (UNA SOLA VEZ)
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [13]:
# 5. Crear Secuencias
features = df[feature_cols].values.astype(np.float32)
target = df[target_col].values.astype(np.float32)

In [14]:
X_seq, y_seq = create_sequences(features, target, window_size)

# 6. Reshape específico para el modelo
if model_type == 'convlstm':
    n_samples, timesteps, n_features = X_seq.shape
    X_seq = X_seq.reshape((n_samples, timesteps, 1, n_features, 1))

if X_seq.size == 0:
    print(f"No se generaron secuencias.")
    # Devolvemos tuplas vacías con la forma correcta
    shape_x = (0, window_size, len(feature_cols)) if model_type == 'transformer' else (0, window_size, 1, len(feature_cols), 1)    

In [15]:
data = {
    "features": X_seq.tolist(),  # lista de lista de listas: [[[...], [...], [...]], ...]
    "label":    y_seq.tolist()
}

features_schema = Features({
    "features": Sequence(Sequence(Value("float64"), length=len(feature_cols)), length=window_size),
    "label":    Value("int64")
})

dataset = Dataset.from_dict(data, features=features_schema)

print(dataset.features)


{'features': List(List(Value('float64'), length=21), length=12), 'label': Value('int64')}


In [ ]:
# Export to JSON
dataset.to_json("../"+output_file_name) # NOTE: Si has movido el script, también hay que arregla aquí

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 21.99ba/s]


16092824

# Test

Vamos a importar el fichero para comprobar si se ha exportado correctamente

In [17]:
# Cargando datos de fichero
dataset_loaded = Dataset.from_json("../"+output_file_name) # NOTE: y aquí, crack

NUM_CLIENTS = 5 # Imagina que tienes dos vacas clientes
partitioner = IidPartitioner(num_partitions=NUM_CLIENTS)
partitioner.dataset = dataset_loaded

# ── 4. Simulate client 0 training ────────────────────────────────
partition = partitioner.load_partition(partition_id=0)
partition_np = partition.with_format("numpy")

X_client = np.stack([partition_np["features"]], axis=1)
y_client = np.stack([partition_np["label"]], axis = 1)


print(f"Client 0 — X shape: {X_client.shape}, y shape: {y_client.shape}")

Generating train split: 0 examples [00:00, ? examples/s]

Failed to load JSON from file '/home/alf/Documents/rewrite_tfg/data/buildingC-data.parquet' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Invalid value. in row 0
Generating train split: 0 examples [00:00, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset